<a href="https://colab.research.google.com/github/0MSA0/supreme-robot/blob/Maximilian_Week_2/attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sample Data:

In [ ]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89, 0.42], # (x^1)
   [0.55, 0.87, 0.66, 0.69], # (x^2)
   [0.57, 0.85, 0.64, 0.91], # (x^3)
   [0.22, 0.58, 0.33, 0.22], # (x^4)
   [0.77, 0.25, 0.10, 0.79], # (x^5)
   [0.05, 0.80, 0.55, 0.19]] # (x^6)
)

d_in = inputs.shape[1] # Input Embedding size
d_out = d_in # Output embedding size

# 2 inputs with 6 tokens of dimension 4
batch = torch.stack((inputs, inputs), dim=0)

Basic Attention:

In [ ]:
import torch.nn as nn

class BasicAttention(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        # Init query, key, and value matrices with random values
        # (On real models, they will be set while training)
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        # Multiply each input with the key matrix to get key vectors
        keys = self.W_key(x)
        # Multiply each input with the query matrix to get query vectors
        queries = self.W_query(x)
        # Multiply each input with the value matrix to get value vectors
        values = self.W_value(x)

        # Multiply all queries with all keys, to determine which words
        # attend to each other (attention scores)
        attn_scores = queries @ keys.transpose(0, 1)
        # Normalize scores via softmax
        attn_weights = torch.softmax(
            attn_scores / d_out**0.5, dim=-1
        )

        # Calculate the context vector bases on
        # normalized attention score an value vectors
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
ba = BasicAttention(d_in, d_out)
print(ba(inputs))

tensor([[ 0.3944, -0.4253,  0.0223, -0.0077],
        [ 0.3946, -0.4250,  0.0210, -0.0072],
        [ 0.3942, -0.4239,  0.0194, -0.0073],
        [ 0.3970, -0.4310,  0.0287, -0.0068],
        [ 0.3947, -0.4249,  0.0208, -0.0075],
        [ 0.3972, -0.4316,  0.0295, -0.0066]], grad_fn=<MmBackward0>)


Causal Attention:

In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        # Pretty much the same as above
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # NEW: Add dropout
        # (Prevents overfitting while training,
        # should be 0 outside of the training)
        self.dropout = nn.Dropout(dropout)

        # NEW: Add mask to prevent the model later "from watching the future"
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        # Pretty much the same as above, but we work in batches now
        # The sequences in a batch can be processed in parallel
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Pretty much the same as above
        attn_scores = queries @ keys.transpose(1, 2)

        # Apply the mask on the attention scores
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # Normalize via softmax and add dropout
        attn_weights = torch.softmax(attn_scores / d_out**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Calculate the context_vectors
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[ 0.4668, -0.5029,  0.0781, -0.0510],
         [ 0.4980, -0.5357,  0.0443, -0.0275],
         [ 0.5004, -0.5432,  0.0694, -0.0143],
         [ 0.4445, -0.4747,  0.0284, -0.0125],
         [ 0.3954, -0.4494,  0.0631, -0.0051],
         [ 0.3972, -0.4316,  0.0295, -0.0066]],

        [[ 0.4668, -0.5029,  0.0781, -0.0510],
         [ 0.4980, -0.5357,  0.0443, -0.0275],
         [ 0.5004, -0.5432,  0.0694, -0.0143],
         [ 0.4445, -0.4747,  0.0284, -0.0125],
         [ 0.3954, -0.4494,  0.0631, -0.0051],
         [ 0.3972, -0.4316,  0.0295, -0.0066]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


Multi Head Attention:

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # NEW: Now we work with multiple heads
        # the token dimensions must be divisible by the number of heads,
        # so that every heads works on an equal part of the token vectors
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        # NEW: Calculate dimensions per head
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        # Pretty much the same as above
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # Pretty much the same as above, but now with multiple heads
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Key, Query and value multiplication with multiple heads in parallel
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # NEW: Transpose the matrix from (b, num_tokens, num_heads, head_dim)
        # to (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute the attention scores in parallel
        # Output: (b, num_heads, num_tokens, head_dim)
        attn_scores = queries @ keys.transpose(2, 3)

        # Pretty much the same as above
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Calculate the context vectors
        context_vec = attn_weights @ values

        # NEW: Bring 'num_heads' back to index 2 (b, num_tokens, num_heads, head_dim)
        # and flatten the head dimensions back to the original output dimension (b, num_tokens, d_out).
        # Then we project the result through a linear layer (out_proj).
        # This mixes/combines the output features of the different heads
        context_vec = context_vec.transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec

torch.manual_seed(420)

batch_size, context_length, d_in = batch.shape
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.7259,  0.6044, -0.2912,  0.5152],
         [-0.6825,  0.5526, -0.1538,  0.4662],
         [-0.6475,  0.5117, -0.0996,  0.4388],
         [-0.6267,  0.4784, -0.1034,  0.4393],
         [-0.5804,  0.3760, -0.0960,  0.4391],
         [-0.5918,  0.4162, -0.1003,  0.4389]],

        [[-0.7259,  0.6044, -0.2912,  0.5152],
         [-0.6825,  0.5526, -0.1538,  0.4662],
         [-0.6475,  0.5117, -0.0996,  0.4388],
         [-0.6267,  0.4784, -0.1034,  0.4393],
         [-0.5804,  0.3760, -0.0960,  0.4391],
         [-0.5918,  0.4162, -0.1003,  0.4389]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


Flexible Multi Head Attention:

In [ ]:
class FlexibleMultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, d_k, context_length, temperature, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # NEW: Ensure d_k is divisible by num_heads instead of d_out
        assert (d_k % num_heads == 0), \
            "d_k must be divisible by num_heads"

        # NEW: d_k and temperature as parameters
        self.d_out = d_out
        self.d_k = d_k
        self.temperature = temperature
        self.num_heads = num_heads

        # NEW: Calculate head_dim based on d_k
        self.head_dim = d_k // num_heads

        # NEW: Project key, query and value to d_k instead of d_out
        self.W_key = nn.Linear(d_in, d_k, bias=qkv_bias)
        self.W_query = nn.Linear(d_in, d_k, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_k, bias=qkv_bias)

        # Pretty much the same as above
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

        # Layer for projecting d_k back to output dimension d_out
        self.out_proj = nn.Linear(d_k, d_out)

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # Pretty much the same as above
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Pretty much the same as above
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # NEW: Softmax with temperatur
        attn_weights = torch.softmax(attn_scores / (self.head_dim**0.5 * self.temperature), dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Calculate the context vectors
        context_vec = attn_weights @ values

        # NEW: Bring 'num_heads' back to index 2 (b, num_tokens, num_heads, head_dim)
        # and flatten the head dimensions back to d_k (b, num_tokens, d_k).
        context_vec = context_vec.transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_k)

        # NEW: Project back to d_out
        context_vec = self.out_proj(context_vec)
        return context_vec

torch.manual_seed(420)

d_k = 8
temperature = 0.5
batch_size, context_length, d_in = batch.shape
mha = FlexibleMultiHeadAttention(d_in, d_out, d_k, context_length, temperature, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[ 0.0239, -0.1606, -0.0631,  0.0249],
         [ 0.1272, -0.2393, -0.0564,  0.0085],
         [ 0.1635, -0.2775, -0.0466, -0.0109],
         [ 0.1439, -0.2701, -0.0564,  0.0146],
         [ 0.1225, -0.2468, -0.0378, -0.0297],
         [ 0.1216, -0.2621, -0.0460,  0.0025]],

        [[ 0.0239, -0.1606, -0.0631,  0.0249],
         [ 0.1272, -0.2393, -0.0564,  0.0085],
         [ 0.1635, -0.2775, -0.0466, -0.0109],
         [ 0.1439, -0.2701, -0.0564,  0.0146],
         [ 0.1225, -0.2468, -0.0378, -0.0297],
         [ 0.1216, -0.2621, -0.0460,  0.0025]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])
